# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShardhaBatra/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

A content page should be reviewed if it receives high search impressions but has poor search performance or low user engagement. My rule prioritizes pages that are still visible in search results but are not attracting enough clicks, have poor average search position, or receive fewer user sessions. These pages are likely good candidates for content improvement or SEO optimization.

## Reason Codes

| Reason Code | Meaning |
|--------------|---------|
| HIGH_IMPRESSIONS | The page receives many search impressions and has the potential to attract more traffic. |
| LOW_CLICKS | The page receives fewer clicks than expected from its impressions. |
| POOR_POSITION | The page has a poor average Google search position. |
| LOW_SESSIONS | The page receives low GA4 sessions, indicating low user visits. |
| LOW_ENGAGEMENT | The page has low user engagement based on scroll events. |

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

login(userdata.get("HF_TOKEN"))

ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train",
    token=userdata.get("HF_TOKEN")
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
import pandas as pd

df = ds.to_pandas()

print(df.shape)

(9841378, 30)


In [5]:
baseline_df = df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "scroll_events"
    ]
].copy()

baseline_df.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN


In [6]:
# Fill missing values

baseline_df["gsc_avg_position"] = baseline_df["gsc_avg_position"].fillna(0)
baseline_df["ga4_sessions"] = baseline_df["ga4_sessions"].fillna(0)
baseline_df["scroll_events"] = baseline_df["scroll_events"].fillna(0)

baseline_df.isnull().sum()

,0
report_date,0
client_hash_id,0
content_hash_id,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_sessions,0
scroll_events,0


In [7]:
# Create a rule-based score

baseline_df["score"] = 0

baseline_df.loc[baseline_df["gsc_impressions"] >= 100, "score"] += 1
baseline_df.loc[baseline_df["gsc_clicks"] <= 5, "score"] += 1
baseline_df.loc[baseline_df["gsc_avg_position"] >= 20, "score"] += 1
baseline_df.loc[baseline_df["ga4_sessions"] <= 10, "score"] += 1
baseline_df.loc[baseline_df["scroll_events"] <= 5, "score"] += 1

baseline_df[["score"]].head()

,score
0,3
1,3
2,4
3,3
4,3


In [8]:
def action(score):
    if score >= 4:
        return "Review Immediately"
    elif score >= 2:
        return "Monitor"
    else:
        return "No Action"

baseline_df["action"] = baseline_df["score"].apply(action)

baseline_df[["score", "action"]].head()

,score,action
0,3,Monitor
1,3,Monitor
2,4,Review Immediately
3,3,Monitor
4,3,Monitor


In [9]:
def reason(row):

    if row["gsc_avg_position"] >= 20:
        return "POOR_POSITION"

    elif row["ga4_sessions"] <= 10:
        return "LOW_SESSIONS"

    elif row["gsc_clicks"] <= 5:
        return "LOW_CLICKS"

    elif row["scroll_events"] <= 5:
        return "LOW_ENGAGEMENT"

    else:
        return "HIGH_IMPRESSIONS"

baseline_df["reason_code"] = baseline_df.apply(reason, axis=1)

baseline_df[["score", "action", "reason_code"]].head()

,score,action,reason_code
0,3,Monitor,LOW_SESSIONS
1,3,Monitor,LOW_SESSIONS
2,4,Review Immediately,LOW_SESSIONS
3,3,Monitor,LOW_SESSIONS
4,3,Monitor,LOW_SESSIONS


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.